# Calculate geography and basic outcomes by LSOA

__Results__:

For the base scenario and with adding MT to both Taunton and Worcester, and for travel bias and not, save these files:
+ `results_geo_outcomes_{scenario_name}.csv`

## Code setup

In [1]:
import os
import numpy as np
import pandas as pd

from classes.geography_processing import Geoprocessing
from classes.model import Model
from classes.scenario import Scenario

In [2]:
dir_output = 'output'
file_scen_params = 'scenario_parameters.csv'

## Load data

Scenario parameters:

In [3]:
p = os.path.join(dir_output, file_scen_params)

# Use converters to turn units column from strings to lists.
# Assumes that no postcodes contain spaces!
converters = {
    'new_mt_units': lambda x: (
        x.replace("'", '').replace(',', '')[1:-1].split()),
}
df_scen = pd.read_csv(p, converters=converters)

In [4]:
display(df_scen.head(6))

,scenario_name,scenario_geoprocess_name,new_mt_units,single_unit_changed,mt_travel_bias_minutes
0,base,base,[],0,0.1
1,taunworc,taunworc,"[TA15DA, WR51DD]",0,0.1
2,only_RG15AN,only_RG15AN,[RG15AN],1,0.1
3,only_HP112TT,only_HP112TT,[HP112TT],1,0.1
4,only_CH21UL,only_CH21UL,[CH21UL],1,0.1
5,only_L355DR,only_L355DR,[L355DR],1,0.1


Split into scenarios to calculate in full and scenarios to only calculate as summary:

In [5]:
m = df_scen['single_unit_changed'] == 0
df_scen_full = df_scen.loc[m].drop('single_unit_changed', axis='columns')
df_scen_sing = df_scen.loc[~m].drop('single_unit_changed', axis='columns')

In [6]:
display(df_scen_full)

,scenario_name,scenario_geoprocess_name,new_mt_units,mt_travel_bias_minutes
0,base,base,[],0.1
1,taunworc,taunworc,"[TA15DA, WR51DD]",0.1


## Main calculations

In [7]:
def calculate_geo_and_outcomes(
        mt_travel_bias,
        new_mt_units,
    ):
    # Process geographic data (only needed when hospital data changes):
    geo = Geoprocessing(limit_to_england=True,
                        mt_travel_bias=mt_travel_bias,
                        new_mt_centres=new_mt_units)
    geo.run()
    geodata = geo.combined_data.copy().reset_index()
    
    # Set up scenario
    scenario = Scenario({
        'name': 'temporary_name',
        'limit_to_england' : True
    })
    
    # Set up model
    model = Model(
        scenario=scenario,
        geodata=geodata
    )
    
    # Run model
    model.run()
    results = model.full_results
    return results

## Run for all scenarios: main changes

Save a copy of the full results to file.

For each set of changed units, calculate the new geography and outcomes data. Then for each set of stroke population parameters, combine the outcomes into the full population.

In [8]:
for scen_geo_name, df in df_scen_full.groupby('scenario_geoprocess_name'):
    # These parameters should be the same for all df in group:
    mt_travel_bias = df.iloc[0]['mt_travel_bias_minutes']
    new_mt_units = df.iloc[0]['new_mt_units']

    # Shared geography and outcome results:
    results = calculate_geo_and_outcomes(
        mt_travel_bias,
        new_mt_units,
    )
    # Round outcome results:
    cols_mrs = [c for c in results.columns if (
                ('mrs' in c) | ('improvement' in c))]
    cols_util = [c for c in results.columns if ('util' in c)]
    results[cols_mrs] = results[cols_mrs].round(3)
    results[cols_util] = results[cols_util].round(5)

    # Save results:
    f = f'results_geo_outcomes_{scen_geo_name}.csv'
    results.to_csv(os.path.join(dir_output, f))

The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.


## Conclusion

Calculated full results, descriptive stats, and stats by unit and saved to file.